# Data Preparation & Feature Engineering
This notebook covers data loading, exploration, location/user/temporal feature extraction, and data cleanup for tourism reviews.

In [1]:
import pandas as pd
import re
import sys
import subprocess
from dataclasses import dataclass
from typing import Optional

df = pd.read_csv("Reviews.csv", encoding='latin1')
# Save a working copy for all processing
df.to_csv("Processed_Reviews.csv", index=False)
# All further processing should use Processed_Reviews.csv
df = pd.read_csv("Processed_Reviews.csv")

# Basic inspection
print("Dataset Overview:")
print(f"  Shape: {df.shape}")
print(f"  Columns: {df.columns.tolist()}")
print(f"\n Data Types:")
print(df.dtypes)
print(f"\n Missing Values:")
print(df.isnull().sum())
print(f"\n Duplicates: {df.duplicated().sum()}")
print(f"\n Rating Distribution:")
print(df["Rating"].value_counts().sort_index())
print(f"\n Top 10 Cities:")
print(df["Located_City"].value_counts().head(10))

Dataset Overview:
  Shape: (16156, 14)
  Columns: ['Location_Name', 'Located_City', 'Location', 'Location_Type', 'User_ID', 'User_Location', 'User_Locale', 'User_Contributions', 'Travel_Date', 'Published_Date', 'Rating', 'Helpful_Votes', 'Title', 'Text']

 Data Types:
Location_Name           str
Located_City            str
Location                str
Location_Type           str
User_ID                 str
User_Location           str
User_Locale             str
User_Contributions    int64
Travel_Date             str
Published_Date          str
Rating                int64
Helpful_Votes         int64
Title                   str
Text                    str
dtype: object

 Missing Values:
Location_Name         0
Located_City          0
Location              0
Location_Type         0
User_ID               0
User_Location         0
User_Locale           0
User_Contributions    0
Travel_Date           0
Published_Date        0
Rating                0
Helpful_Votes         0
Title              

In [2]:


provinces = [
    "Western Province", "Central Province", "Southern Province",
    "Northern Province", "Eastern Province", "North Western Province",
    "North Central Province", "Uva Province", "Sabaragamuwa Province",
]

city_to_district = {
    "Arugam Bay": "Ampara",
    "Ampara": "Ampara",
    "Galle": "Galle",
    "Kalutara": "Kalutara",
    "Trincomalee": "Trincomalee",
    "Matara": "Matara",
    "Colombo": "Colombo",
    "Gampaha": "Gampaha",
    "Nilaveli": "Trincomalee",
    "Kalkudah": "Batticaloa",
    "Nuwara Eliya": "Nuwara Eliya",
    "Kandy": "Kandy",
    "Tissamaharama": "Hambantota",
    "Anuradhapura": "Anuradhapura",
    "Haputale": "Badulla",
    "Badulla": "Badulla",
    "Beruwala": "Kalutara",
    "Jaffna": "Jaffna",
    "Polonnaruwa": "Polonnaruwa",
    "Matale": "Matale",
    "Weligatta": "Hambantota",
    "Ratnapura": "Ratnapura",
    "Tangalle": "Hambantota",
    "Pinnawala": "Kegalle",
    "Deniyaya": "Matara",
    "Koslanda": "Badulla",
    "Mirissa": "Matara",
    "Ella": "Badulla",
    "Negombo": "Gampaha",
    "Sigiriya": "Matale",
    "Batticaloa": "Batticaloa",
    "Bentota": "Galle",
    "Hikkaduwa": "Galle",
    "Mirigama": "Gampaha",
    "Habarana": "Anuradhapura",
}

manual_location_mappings = {
    "Udawalawe National Park": {"province": "Sabaragamuwa Province", "district": "Ratnapura"},
    "North Central Province": {"province": "North Central Province", "district": "Anuradhapura"},
}

province_pattern = re.compile(
    r"(" + "|".join([re.escape(p) for p in provinces]) + r")", flags=re.IGNORECASE
)

def extract_province(location_str):
    """Extract province from location string"""
    if not isinstance(location_str, str) or not location_str.strip():
        return None
    
    loc = location_str.strip()
    
    # Manual mapping
    for k, v in manual_location_mappings.items():
        if k.lower() == loc.lower():
            return v.get("province")
    
    # Pattern matching
    m = province_pattern.search(loc)
    if m:
        return m.group(1)
    
    # Numeric pattern
    m2 = re.search(r"([A-Za-z ]+Province)\b", loc)
    if m2:
        return m2.group(1).strip()
    
    return None

def infer_district(row):
    """Infer district from location and city"""
    city = row.get("Located_City")
    location = row.get("Location")
    
    # Manual mapping
    if isinstance(location, str):
        for k, v in manual_location_mappings.items():
            if k.lower() == location.strip().lower():
                return v.get("district")
    
    # City mapping
    if isinstance(city, str) and city in city_to_district:
        return city_to_district[city]
    
    if not isinstance(location, str):
        return None
    
    parts = [p.strip() for p in location.split(",") if p.strip()]
    
    # Check for explicit district
    for part in parts:
        if "District" in part:
            return part.replace("District", "").strip()
    
    # Infer from position
    if len(parts) >= 3:
        return parts[-2]
    elif len(parts) == 2:
        return parts[0]
    elif len(parts) == 1 and not any(parts[0].lower() == p.lower() for p in provinces):
        return parts[0]
    
    return None

# Apply location extraction
print(" Extracting province and district...")
df["Province"] = df["Location"].apply(extract_province)
df["District"] = df.apply(infer_district, axis=1)

print(f"Province coverage: {df['Province'].notna().sum()}/{len(df)} rows")
print(f"District coverage: {df['District'].notna().sum()}/{len(df)} rows")
print(f"\n Top Provinces:")
print(df["Province"].value_counts().head(10))

 Extracting province and district...
Province coverage: 16156/16156 rows
District coverage: 16156/16156 rows

 Top Provinces:
Province
Central Province          5252
North Central Province    3171
Southern Province         2648
Western Province          1890
Eastern Province          1162
Uva Province              1040
Sabaragamuwa Province      518
Northern Province          475
Name: count, dtype: int64


In [3]:
# Install pycountry if needed
try:
    import pycountry
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pycountry"])
    import pycountry

@dataclass
class ParseResult:
    country: Optional[str]
    other: Optional[str]
    method: str
    confidence: float

# Regional data
US_STATES = {"alabama", "alaska", "arizona", "arkansas", "california", "colorado",
             "connecticut", "delaware", "florida", "georgia", "hawaii", "idaho",
             "illinois", "indiana", "iowa", "kansas", "kentucky", "louisiana",
             "maine", "maryland", "massachusetts", "michigan", "minnesota",
             "mississippi", "missouri", "montana", "nebraska", "nevada",
             "new hampshire", "new jersey", "new mexico", "new york",
             "north carolina", "north dakota", "ohio", "oklahoma", "oregon",
             "pennsylvania", "rhode island", "south carolina", "south dakota",
             "tennessee", "texas", "utah", "vermont", "virginia", "washington",
             "west virginia", "wisconsin", "wyoming", "district of columbia", "puerto rico"}

AU_STATES = {"new south wales", "queensland", "south australia", "tasmania",
             "victoria", "western australia", "australian capital territory", "northern territory"}

CA_PROVINCES = {"alberta", "british columbia", "manitoba", "new brunswick",
                "newfoundland and labrador", "nova scotia", "ontario",
                "prince edward island", "quebec", "saskatchewan",
                "northwest territories", "nunavut", "yukon"}

INDIA_STATES = {"andhra pradesh", "arunachal pradesh", "assam", "bihar",
                "chhattisgarh", "goa", "gujarat", "haryana", "himachal pradesh",
                "jharkhand", "karnataka", "kerala", "madhya pradesh",
                "maharashtra", "manipur", "meghalaya", "mizoram", "nagaland",
                "odisha", "punjab", "rajasthan", "sikkim", "tamil nadu",
                "telangana", "tripura", "uttar pradesh", "uttarakhand",
                "west bengal", "delhi", "jammu and kashmir", "ladakh"}

UK_REGIONS = {"england", "scotland", "wales", "northern ireland"}

REGION_TO_COUNTRY = {
    **{state: "United States" for state in US_STATES},
    **{state: "Australia" for state in AU_STATES},
    **{state: "Canada" for state in CA_PROVINCES},
    **{state: "India" for state in INDIA_STATES},
    **{region: "United Kingdom" for region in UK_REGIONS},
    "new england": "United States",
}

# Build country index
def build_country_index():
    alias_map = {}
    for country in pycountry.countries:
        alias_map[country.name.lower()] = country.name
    
    manual = {
        "usa": "United States", "u.s.a": "United States", "us": "United States",
        "u.s": "United States", "uk": "United Kingdom", "u.k": "United Kingdom",
        "uae": "United Arab Emirates", "u.a.e": "United Arab Emirates",
        "russia": "Russian Federation", "viet nam": "Vietnam",
    }
    alias_map.update(manual)
    
    preferred = {
        "Korea, Republic of": "South Korea",
        "Korea, Democratic People's Republic of": "North Korea",
        "Russian Federation": "Russia", "Viet Nam": "Vietnam",
    }
    return alias_map, preferred

COUNTRY_ALIAS_TO_NAME, COUNTRY_SHORT_TO_PREFERRED = build_country_index()

def parse_user_location(raw_value):
    """Parse user location to extract country and region"""
    if raw_value is None or (isinstance(raw_value, float) and pd.isna(raw_value)):
        return ParseResult(country=None, other=None, method="empty", confidence=0.0)
    
    text = str(raw_value).strip()
    if not text:
        return ParseResult(country=None, other=None, method="empty", confidence=0.0)
    
    parts = [p.strip() for p in text.split(",") if p.strip()]
    country = None
    
    # Search for country
    for part in reversed(parts):
        if part.lower() in COUNTRY_ALIAS_TO_NAME:
            country = COUNTRY_SHORT_TO_PREFERRED.get(
                COUNTRY_ALIAS_TO_NAME[part.lower()],
                COUNTRY_ALIAS_TO_NAME[part.lower()]
            )
            break
    
    # Search for region
    region = None
    for part in reversed(parts):
        if part.lower() in REGION_TO_COUNTRY:
            region = part.title()
            if not country:
                country = REGION_TO_COUNTRY[part.lower()]
            break
    
    if country:
        return ParseResult(
            country=country, other=region,
            method="rule_based", confidence=0.9 if region else 0.85
        )
    
    return ParseResult(country=None, other=None, method="unresolved", confidence=0.0)

# Apply location parsing
print(" Parsing user locations...")
parsed = df["User_Location"].apply(parse_user_location) # type: ignore
df["User_Country"] = parsed.apply(lambda x: x.country)
df["User_Region"] = parsed.apply(lambda x: x.other)

print(f"Country resolution: {df['User_Country'].notna().sum()}/{len(df)} rows")
print(f"\n Top 15 Countries:")
print(df["User_Country"].value_counts().head(15))

 Parsing user locations...
Country resolution: 14494/16156 rows

 Top 15 Countries:
User_Country
United Kingdom          4530
Australia               2000
India                   1625
United States            950
United Arab Emirates     432
Canada                   396
Singapore                340
Germany                  307
New Zealand              240
France                   233
China                    214
Malaysia                 202
Ireland                  142
Belgium                  142
Switzerland              140
Name: count, dtype: int64


In [4]:
# Convert dates
for col in ['Travel_Date', 'Published_Date']:
    df[col] = pd.to_datetime(df[col], errors='coerce', utc=True)
    df[f'{col}_Month'] = df[col].dt.month
    df[f'{col}_Year'] = df[col].dt.year

# Add text features
df['Review_Length'] = df['Text'].apply(len)
df['Title_Length'] = df['Title'].apply(len)

print("Temporal features extracted")
print(f"\n Date Range:")
print(f"  Travel: {df['Travel_Date'].min()} to {df['Travel_Date'].max()}")
print(f"  Published: {df['Published_Date'].min()} to {df['Published_Date'].max()}")

Temporal features extracted

 Date Range:
  Travel: 2010-09-01 00:00:00+00:00 to 2023-05-01 00:00:00+00:00
  Published: 2011-03-12 12:00:09+00:00 to 2023-05-20 05:15:38+00:00


In [5]:
# --- Add Review Delay and Rating Class Columns ---
def get_rating_class(r):
    if r <= 2:
        return "Negative"
    elif r == 3:
        return "Neutral"
    else:
        return "Positive"

df["Rating_Class"] = df["Rating"].apply(get_rating_class)

if "Published_Date" in df.columns and "Travel_Date" in df.columns:
    df["Review_Delay_Days"] = (df["Published_Date"] - df["Travel_Date"]).dt.days
    # Fix negative values (bad data)
    df["Review_Delay_Days"] = df["Review_Delay_Days"].clip(lower=0)
else:
    df["Review_Delay_Days"] = None

print("Added Review_Delay_Days and Rating_Class columns (negative delays fixed)")

Added Review_Delay_Days and Rating_Class columns (negative delays fixed)


In [6]:
# Drop raw location columns (redundant with extracted features)
df = df.drop(columns=['Location', 'User_Location'], errors='ignore')
# Drop 'travel_date' and 'published_date' columns if they exist
for col in ['Travel_Date', 'Published_Date']:
    if col in df.columns:
        df = df.drop(col, axis=1)
        print(f"'{col}' column dropped.")
    else:
        print(f"'{col}' column not found.")
# Drop 'User_ID' column if it exists
if 'User_ID' in df.columns:
    df = df.drop('User_ID', axis=1)
    print("'User_ID' column dropped.")
else:
    print("'User_ID' column not found.")

# Reorder columns as requested
ordered_cols = [
    'Location_Name', 'Located_City', 'Province', 'District', 'Location_Type',
    'User_Locale', 'User_Country', 'User_Region', 'Travel_Date_Month', 'Travel_Date_Year',
    'Published_Date_Month', 'Published_Date_Year',
    'User_Contributions', 'Rating', 'Helpful_Votes',
    'Title', 'Text', 'Review_Length', 'Title_Length', 'Rating_Class', 'Review_Delay_Days'
   
]
# Only keep columns that exist in the DataFrame
ordered_cols = [col for col in ordered_cols if col in df.columns]
df = df[ordered_cols + [col for col in df.columns if col not in ordered_cols]]

print(f"Cleaned up redundant columns and reordered")
print(f"\n Dataset Shape: {df.shape}")
print(f"\n Current Columns ({len(df.columns)}):")
for i, col in enumerate(df.columns, 1):
    print(f"  {i:2d}. {col}")
df.to_csv("Processed_Reviews.csv")

'Travel_Date' column dropped.
'Published_Date' column dropped.
'User_ID' column dropped.
Cleaned up redundant columns and reordered

 Dataset Shape: (16156, 21)

 Current Columns (21):
   1. Location_Name
   2. Located_City
   3. Province
   4. District
   5. Location_Type
   6. User_Locale
   7. User_Country
   8. User_Region
   9. Travel_Date_Month
  10. Travel_Date_Year
  11. Published_Date_Month
  12. Published_Date_Year
  13. User_Contributions
  14. Rating
  15. Helpful_Votes
  16. Title
  17. Text
  18. Review_Length
  19. Title_Length
  20. Rating_Class
  21. Review_Delay_Days


In [15]:
df["District"].unique().tolist()

['Ampara',
 'Galle',
 'Kalutara',
 'Trincomalee',
 'Matara',
 'Colombo',
 'Gampaha',
 'Batticaloa',
 'Nuwara Eliya',
 'Kandy',
 'Hambantota',
 'Anuradhapura',
 'Badulla',
 'Jaffna',
 'Polonnaruwa',
 'Matale',
 'Ratnapura',
 'Tangalle',
 'Kegalle']